In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
subset_size = 300
dataset = dataset.select(range(min(subset_size, len(dataset))))
print("Dataset split: glue/mrpc validation")
print(f"Subset size: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = dataset["label"]

emb1 = encode_texts(sentence1_list, batch_size=64, max_length=128)
emb2 = encode_texts(sentence2_list, batch_size=64, max_length=128)
cosine_similarities = F.cosine_similarity(emb1, emb2).tolist()

print(f"Completed embedding inference for {len(cosine_similarities)} examples.")

In [ ]:
threshold_grid = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

best_threshold = None
best_f1 = -1.0
best_predictions = None

for threshold in threshold_grid:
    predictions = [1 if score >= threshold else 0 for score in cosine_similarities]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_predictions = predictions

predictions = best_predictions
threshold = best_threshold
confidences = [abs(score - threshold) for score in cosine_similarities]

print(f"Selected best threshold from fixed grid: {threshold}")
print(f"Best subset F1 on threshold grid: {best_f1:.4f}")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

positive_scores = [score for score, label in zip(cosine_similarities, labels) if label == 1]
negative_scores = [score for score, label in zip(cosine_similarities, labels) if label == 0]
mean_positive_similarity = sum(positive_scores) / len(positive_scores)
mean_negative_similarity = sum(negative_scores) / len(negative_scores)

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print(f"Mean cosine similarity | label=1: {mean_positive_similarity:.4f}")
print(f"Mean cosine similarity | label=0: {mean_negative_similarity:.4f}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
num_examples_to_show = 8

false_positive_indices = [i for i, (y_true, y_pred) in enumerate(zip(labels, predictions)) if y_true == 0 and y_pred == 1]
false_negative_indices = [i for i, (y_true, y_pred) in enumerate(zip(labels, predictions)) if y_true == 1 and y_pred == 0]

false_positive_indices = sorted(false_positive_indices, key=lambda i: cosine_similarities[i], reverse=True)
false_negative_indices = sorted(false_negative_indices, key=lambda i: cosine_similarities[i])

print("TOP FALSE POSITIVES")
for rank, i in enumerate(false_positive_indices[:num_examples_to_show], start=1):
    row = dataset[i]
    print(f"False positive {rank}")
    print(f"index: {i}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {labels[i]} ({label_map[labels[i]]})")
    print(f"pred label: {predictions[i]} ({label_map[predictions[i]]})")
    print(f"cosine similarity: {cosine_similarities[i]:.4f}")
    print(f"distance above threshold: {cosine_similarities[i] - threshold:.4f}")
    print("-" * 80)

print("TOP FALSE NEGATIVES")
for rank, i in enumerate(false_negative_indices[:num_examples_to_show], start=1):
    row = dataset[i]
    print(f"False negative {rank}")
    print(f"index: {i}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {labels[i]} ({label_map[labels[i]]})")
    print(f"pred label: {predictions[i]} ({label_map[predictions[i]]})")
    print(f"cosine similarity: {cosine_similarities[i]:.4f}")
    print(f"distance below threshold: {threshold - cosine_similarities[i]:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("inference_method=separate_sentence_embeddings_with_cosine_similarity")
print("dataset_split=glue/mrpc validation")
print(f"subset_size={len(dataset)}")
print(f"device={device}")
print(f"threshold_selection=fixed_grid_single_pass_on_same_subset")
print(f"threshold_grid={threshold_grid}")
print(f"selected_threshold={threshold}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_positive_similarity={mean_positive_similarity:.4f}")
print(f"mean_negative_similarity={mean_negative_similarity:.4f}")
print(f"num_false_positives={len(false_positive_indices)}")
print(f"num_false_negatives={len(false_negative_indices)}")